# UK Road Collisions: model training and evaluation

Predict KSI using only information available at or before the collision, with a future-facing time split.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from road_severity.data import build_features, load_collisions, make_target
from road_severity.modeling import evaluate, make_pipeline, save_model_outputs, temporal_split

config = yaml.safe_load((ROOT / 'configs/default.yaml').read_text(encoding='utf-8'))
DATA_PATH = ROOT / config['data']['raw_path']
MAX_ROWS = config['project']['max_rows']  # Set to 0 for all rows.
config

## Temporal split

Train on years before 2024, use 2024 for validation, and reserve 2025 for final testing.

In [ ]:
frame = load_collisions(DATA_PATH, None if MAX_ROWS == 0 else MAX_ROWS)
train, validation, test = temporal_split(frame, config['model']['validation_year'], config['model']['test_year'])
pd.DataFrame({'split': ['train', 'validation', 'test'], 'rows': [len(train), len(validation), len(test)], 'period': ['before 2024', '2024', '2025']})

In [ ]:
task = config['model']['task']
X_train, y_train = build_features(train), make_target(train, task)
X_validation, y_validation = build_features(validation), make_target(validation, task)
X_test, y_test = build_features(test), make_target(test, task)
pd.DataFrame({'split': ['train', 'validation', 'test'], 'KSI prevalence': [y_train.mean(), y_validation.mean(), y_test.mean()]})

## Train and validate

The pipeline handles missing values and categorical encoding. `build_features` excludes outcome-derived leakage fields.

In [ ]:
model = make_pipeline(X_train, config['project']['random_state'], config['model'])
model.fit(X_train, y_train)
validation_metrics = evaluate(model, X_validation, y_validation)
{key: value for key, value in validation_metrics.items() if key != 'classification_report'}

## Final test and feature importance

Run after fixing the approach. Outputs are stored locally under `models/test/` and are ignored by Git.

In [ ]:
test_metrics = save_model_outputs(model, X_test, y_test, ROOT / 'models/test')
{key: value for key, value in test_metrics.items() if key != 'classification_report'}

In [ ]:
importance = pd.read_csv(ROOT / 'models/test/permutation_importance.csv')
importance.head(15)

## Interpretation checklist

Compare Average Precision with KSI prevalence, inspect error patterns before selecting a threshold, and describe importance as predictive association rather than causation.